# Machine Learning vs Deep Learning — Concepts & Side-by-Side Implementation

This single notebook covers **both** fields in one place:

- **Part A — Machine Learning**: Concepts, types, and a hands-on **Random Forest** classifier on the **Iris dataset** (easy to explain, tabular data).
- **Part B — Deep Learning**: Concepts, types, and a hands-on **CNN (Convolutional Neural Network)** on the **MNIST dataset** (harder to explain, raw image data).
- **Part C — Final Comparison**: A direct, side-by-side comparison of both approaches, including a live runtime/accuracy comparison chart built from the actual results of Part A and Part B.

> **Tip:** In Colab, go to `Runtime > Change runtime type > GPU` before running, so Part B (deep learning) trains faster.

---
# Part A — Machine Learning

## A.1 What is Machine Learning?

**Machine Learning (ML)** is a branch of Artificial Intelligence where a system learns patterns from data
and improves its performance on a task **without being explicitly programmed with fixed rules**.

Instead of hand-coding rules (`if... else...`), we give the algorithm labeled examples, and it learns a
mathematical function mapping inputs to outputs.

```
Data (X, y)  --->  Learning Algorithm  --->  Model (learned function f)
New Input X  --->  f(X)  --->  Prediction
```

### Types of Machine Learning

| Type | Description | Example |
|---|---|---|
| **Supervised Learning** | Learns from **labeled** data | Spam detection, Iris classification *(this notebook)* |
| **Unsupervised Learning** | Learns from **unlabeled** data, finds hidden structure | Customer segmentation, clustering |
| **Semi-Supervised Learning** | Mix of small labeled + large unlabeled data | Medical image labeling with few experts |
| **Reinforcement Learning** | Learns via **trial and error** using rewards | Game playing (AlphaGo), robotics |

Supervised learning splits further into:
- **Classification** — predicting a category (used here)
- **Regression** — predicting a continuous number

### Common ML Algorithms
Linear/Logistic Regression, Decision Trees, **Random Forest** (used here), SVM, KNN, Naive Bayes,
Gradient Boosting (XGBoost), K-Means (unsupervised).

## A.2 Why Random Forest?

A **Decision Tree** asks yes/no questions about features to reach a decision — easy to understand, but
can overfit. A **Random Forest** builds **many trees** on random subsets of data/features and combines
their predictions by **majority vote** — more accurate and robust, still intuitive.

> **Analogy:** Instead of one doctor's diagnosis, ask a panel of 100 doctors and go with the majority.

### A.3 Import Libraries

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set(style="whitegrid")
print("ML libraries imported successfully.")

### A.4 Load the Dataset

**Iris Dataset**: 150 samples, 4 numeric features (sepal/petal length & width), 3 classes
(*Setosa, Versicolor, Virginica*). Small, clean, tabular — ideal for classical ML.

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print("Shape of dataset:", df.shape)
df.head()

### A.5 Explore the Data (EDA)

In [ ]:
print(df["species"].value_counts())
sns.pairplot(df, hue="species", corner=True)
plt.suptitle("Iris: Pairwise Feature Relationships by Species", y=1.02)
plt.show()

### A.6 Preprocess — Train/Test Split + Scaling

We split into 80% train / 20% test. Random Forest doesn't strictly need scaling, but we include it
to keep the pipeline general.

In [ ]:
X = df[iris.feature_names]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

### A.7 Train the Random Forest Model
We also time the training — we'll use this later in the final comparison.

In [ ]:
ml_start = time.time()

ml_model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42)
ml_model.fit(X_train_scaled, y_train)

ml_train_time = time.time() - ml_start
print(f"Model trained with {len(ml_model.estimators_)} trees in {ml_train_time:.4f} seconds.")

### A.8 Evaluate the Model

In [ ]:
y_pred_ml = ml_model.predict(X_test_scaled)

ml_accuracy = accuracy_score(y_test, y_pred_ml)
print(f"ML Test Accuracy: {ml_accuracy*100:.2f}%\n")
print(classification_report(y_test, y_pred_ml))

cm = confusion_matrix(y_test, y_pred_ml, labels=ml_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ml_model.classes_)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Random Forest on Iris")
plt.show()

### A.9 Feature Importance
Random Forest lets us directly see which features mattered most — a key interpretability advantage of classical ML.

In [ ]:
importances = pd.Series(ml_model.feature_importances_, index=iris.feature_names).sort_values(ascending=False)

plt.figure(figsize=(7,4))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.show()

importances

---
# Part B — Deep Learning

## B.1 What is Deep Learning?

**Deep Learning (DL)** is a subfield of ML based on **Artificial Neural Networks** with many layers.
Unlike classical ML, DL models **automatically learn features** directly from raw data (pixels, audio,
raw text) instead of relying on hand-crafted features.

```
Raw Input (pixels)
   -> Layer 1 (edges)
   -> Layer 2 (shapes)
   -> Layer 3 (parts)
   -> Output (full concept: "this is a 7")
```

### Types of Deep Learning Architectures

| Type | Description | Common Use Case |
|---|---|---|
| **ANN / MLP** | Fully connected layers of neurons | Simple tabular tasks |
| **CNN** | Filters/kernels detect spatial patterns | Image classification *(this notebook)* |
| **RNN / LSTM / GRU** | Processes sequences, remembers context | Time series, text, speech |
| **Transformer** | "Attention" relates all sequence parts at once | Language models (GPT, BERT) |
| **Autoencoder** | Learns to compress & reconstruct data | Anomaly detection, denoising |
| **GAN** | Generator vs discriminator compete | Image generation |

### Key DL Building Blocks
Neuron/Perceptron, Activation Functions (ReLU, Softmax), Backpropagation, Optimizer (Adam, SGD),
Loss Function (cross-entropy), Epoch/Batch.

## B.2 Why CNN for This Task?

Images have **spatial structure** — nearby pixels are related. A plain fully-connected network ignores
this. A **CNN** slides small filters across the image to detect local patterns (edges → shapes → digits),
similar to how humans visually recognize digits — making it the classic first deep learning project.

### B.3 Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

### B.4 Load the Dataset

**MNIST Dataset**: 60,000 training + 10,000 test images, 28x28 grayscale, digits 0–9.
Unlike Iris's 4 hand-picked features, here the **raw pixels** are the input — the key difference
between classical ML and deep learning.

In [ ]:
(X_train_dl, y_train_dl), (X_test_dl, y_test_dl) = tf.keras.datasets.mnist.load_data()

print("Training data shape:", X_train_dl.shape)
print("Testing data shape:", X_test_dl.shape)

### B.5 Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10,4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_dl[i], cmap="gray")
    ax.set_title(f"Label: {y_train_dl[i]}")
    ax.axis("off")
plt.suptitle("Sample MNIST Digits")
plt.tight_layout()
plt.show()

### B.6 Preprocess — Normalize + Reshape

1. **Normalize** pixels from [0, 255] to [0, 1] for stable/fast training.
2. **Reshape** to `(28, 28, 1)` since Conv2D expects a channel dimension.

In [ ]:
X_train_dl = X_train_dl.astype("float32") / 255.0
X_test_dl = X_test_dl.astype("float32") / 255.0

X_train_dl = X_train_dl.reshape(-1, 28, 28, 1)
X_test_dl = X_test_dl.reshape(-1, 28, 28, 1)

print("New training shape:", X_train_dl.shape)
print("New testing shape:", X_test_dl.shape)

### B.7 Build the CNN Architecture

1. **Conv2D + ReLU** — learns low-level features (edges)
2. **MaxPooling2D** — downsamples, keeps strongest signals
3. **Conv2D + ReLU** (deeper) — learns higher-level shapes
4. **MaxPooling2D**
5. **Flatten** — 2D feature maps -> 1D vector
6. **Dense + ReLU** — combines features
7. **Dropout** — prevents overfitting
8. **Dense (Softmax, 10 units)** — probability per digit

In [ ]:
dl_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

dl_model.summary()

### B.8 Compile and Train

- **Optimizer**: Adam
- **Loss**: Sparse categorical cross-entropy
- **EarlyStopping**: stops training if validation loss stops improving

We time training here too, for the final comparison.

In [ ]:
dl_model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

dl_start = time.time()

history = dl_model.fit(
    X_train_dl, y_train_dl,
    validation_split=0.1,
    epochs=15,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

dl_train_time = time.time() - dl_start
print(f"\nCNN trained in {dl_train_time:.2f} seconds.")

### B.9 Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].plot(history.history["accuracy"], label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

### B.10 Evaluate on Test Data

In [ ]:
dl_test_loss, dl_accuracy = dl_model.evaluate(X_test_dl, y_test_dl, verbose=0)
print(f"DL Test Accuracy: {dl_accuracy*100:.2f}%")
print(f"DL Test Loss: {dl_test_loss:.4f}")

y_pred_probs = dl_model.predict(X_test_dl)
y_pred_dl = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test_dl, y_pred_dl))

In [ ]:
cm_dl = confusion_matrix(y_test_dl, y_pred_dl)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_dl)
fig, ax = plt.subplots(figsize=(8,8))
disp.plot(cmap="Blues", ax=ax, colorbar=False)
plt.title("Confusion Matrix - CNN on MNIST")
plt.show()

### B.11 Visualize Some Predictions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10,4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test_dl[i].reshape(28,28), cmap="gray")
    color = "green" if y_pred_dl[i] == y_test_dl[i] else "red"
    ax.set_title(f"Pred: {y_pred_dl[i]} / True: {y_test_dl[i]}", color=color)
    ax.axis("off")
plt.suptitle("Sample Predictions (Green = Correct, Red = Wrong)")
plt.tight_layout()
plt.show()

---
# Part C — Final Comparison: Machine Learning vs Deep Learning

Now that we've implemented **both** a classical ML model (Random Forest on Iris) and a **Deep Learning**
model (CNN on MNIST) **in the same notebook**, here is a direct comparison — including the actual
numbers produced by the two models you just trained above.

| Aspect | Machine Learning (Random Forest / Iris) | Deep Learning (CNN / MNIST) |
|---|---|---|
| **Dataset size** | Small (150 samples) is enough | Needs large data (60,000+ images) |
| **Input type** | Structured/tabular, hand-picked features | Raw, unstructured data (pixels) |
| **Feature engineering** | Often manual, domain-driven | Automatic — learned via convolution layers |
| **Model complexity** | Simple (trees + voting) | Complex (many layers, many parameters) |
| **Training time** | Fast — seconds on CPU | Slower — benefits from GPU |
| **Hardware needs** | Any laptop CPU | Ideally GPU/TPU |
| **Interpretability** | High — feature importances easy to inspect | Lower — often a "black box" |
| **Best suited for** | Tabular data, smaller datasets, fast prototyping | Images, audio, text, large/high-dimensional data |

### Live Results Comparison
The cell below plots the **actual accuracy and training time** from the two models you trained in this notebook.

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["ML: Random Forest (Iris)", "DL: CNN (MNIST)"],
    "Test Accuracy (%)": [ml_accuracy*100, dl_accuracy*100],
    "Training Time (seconds)": [ml_train_time, dl_train_time]
})

print(comparison_df)

fig, axes = plt.subplots(1, 2, figsize=(12,4.5))

sns.barplot(data=comparison_df, x="Model", y="Test Accuracy (%)", ax=axes[0], palette="Set2")
axes[0].set_title("Test Accuracy Comparison")
axes[0].set_ylim(0, 100)
for i, v in enumerate(comparison_df["Test Accuracy (%)"]):
    axes[0].text(i, v + 1, f"{v:.2f}%", ha="center")

sns.barplot(data=comparison_df, x="Model", y="Training Time (seconds)", ax=axes[1], palette="Set2")
axes[1].set_title("Training Time Comparison")
for i, v in enumerate(comparison_df["Training Time (seconds)"]):
    axes[1].text(i, v, f"{v:.2f}s", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## Key Takeaway

- **Random Forest (ML)** reached very high accuracy on Iris almost instantly, with a small dataset and
  no GPU — and we could directly interpret *why* it made each decision (feature importance).
- **CNN (DL)** needed far more data, more training time, and (ideally) a GPU — but it worked directly on
  **raw pixels** with **no manual feature engineering**, and handled a much harder, higher-dimensional
  problem (10-class image recognition vs 3-class tabular classification).

**Rule of thumb:** Start with classical ML for structured/tabular data or small datasets. Reach for deep
learning when you have large amounts of unstructured data (images, audio, text) where automatic feature
learning outweighs the added cost in data, compute, and interpretability.